# 🔬 ISOM 260: Build a Research Agent with ReAct Reasoning

**Session 6 — Agent Patterns & Production Intelligence** | Suffolk University | Prof. Hasan Arslan

---

## From Session 5 to Session 6: One Paragraph Changes Everything

Last session you connected your agents to the **real world**:

```
AI Agent = LLM + Tools + Reasoning Loop + Memory
```

You gave Claude real tools — Wikipedia search, web page fetching, and a calculator — that work with live APIs. Your agent handles errors and works with real data.

But watch what happens when you ask it to **research** something. It searches once, dumps what it finds, and stops. No evaluation. No planning. No follow-up.

## What's New Today: ReAct Reasoning

**ReAct** = **Re**ason + **Act**

Instead of blindly calling tools, a ReAct agent:

1. 💭 **Thinks** — "What do I need to find out? What's the best tool for this?"
2. 🛠️ **Acts** — Calls a tool with specific inputs
3. 👁️ **Observes** — Reads the result and evaluates quality
4. 🔁 **Repeats** — Decides if it needs more info or is ready to synthesize

This is the exact pattern behind **NanoClaw** (the agent from Session 5), **Claude Code**, **Perplexity**, and every serious AI agent in production today.

**The twist:** you won't write much new code today. You'll take your **exact Session 5 tools**, change **one paragraph** — the system prompt — and watch the behavior transform completely.

---

## 🚀 Part 1: Setup (2 minutes)

Same setup as Sessions 4 and 5 — install the SDK and connect your API key.

In [ ]:
# Install dependencies
# anthropic = Claude SDK (same as Sessions 4-5)
# requests + beautifulsoup4 = for real API tools (same as Session 5)
!pip install -q anthropic requests beautifulsoup4

In [ ]:
# Set your API key
# Same key you've been using since Session 4 — it still works!
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named ANTHROPIC_API_KEY
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
import anthropic
import json

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Session 6 connected!' and nothing else."}]
)

print(f"🟢 {response.content[0].text}")
print(f"   Model: {response.model}")

---

## 🔑 The One-Line Difference

Last session, your `run_agent()` used this system prompt:

> "You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning."

Today, we change **just the system prompt** to this:

> "You are a research agent that uses the ReAct (Reason + Act) framework. For every task:
> 1. THINK: Before each action, explain your reasoning.
> 2. ACT: Use a tool to gather information.
> 3. OBSERVE: Reflect on what you learned.
> 4. REPEAT or CONCLUDE."

**That's it.** Same agent loop. Same tool calling. Same `while True` structure. The code difference is ONE paragraph. But the behavior difference is everything.

This is the exact pattern behind **NanoClaw** (the agent you saw research live in Session 5), **Claude Code**, **Perplexity**, and every serious AI agent in production.

Let's prove it.

---

## 🔧 Your Session 5 Toolkit — Bring It Back

Before we change anything, let's bring back the **exact tools you already built**. This is the same code from Session 5 — real Wikipedia API, real web fetching, real calculator.

We need these working first. Then we'll change **only the system prompt** and watch what happens.

In [ ]:
# ============================================
# YOUR SESSION 5 TOOLS — Same real tools!
# ============================================
# Real Wikipedia API. Real web fetching. Real calculator.
# Copied from your Session 5 notebook — nothing changed.

import requests
from bs4 import BeautifulSoup

# --- Tool 1: Wikipedia Search (REAL API) ---
wikipedia_tool = {
    "name": "wikipedia_search",
    "description": (
        "Search Wikipedia for information about any topic. Returns the article title, "
        "a summary, and a link to the full article. Use this for factual lookups, "
        "background research, and finding reliable information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The topic to search for on Wikipedia"
            }
        },
        "required": ["topic"]
    }
}

def wikipedia_search(topic: str) -> str:
    """Search Wikipedia — REAL API, live data!"""
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 404:
            return json.dumps({"error": f"No Wikipedia page found for '{topic}'. Try a different search term."})
        data = response.json()
        return json.dumps({
            "title": data.get("title", ""),
            "summary": data.get("extract", ""),
            "url": data.get("content_urls", {}).get("desktop", {}).get("page", ""),
        }, indent=2)
    except requests.exceptions.Timeout:
        return json.dumps({"error": "Wikipedia request timed out. Try again."})
    except Exception as e:
        return json.dumps({"error": f"Wikipedia search failed: {str(e)}"})


# --- Tool 2: Web Page Fetcher (REAL) ---
fetch_tool = {
    "name": "fetch_web_page",
    "description": (
        "Fetch a web page and extract its text content. Use this to read articles, "
        "reports, or any web page when you have a specific URL. Returns the page title "
        "and text content (truncated to 3000 characters)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The full URL to fetch (must start with http:// or https://)"
            }
        },
        "required": ["url"]
    }
}

def fetch_web_page(url: str) -> str:
    """Fetch and extract text from a web page — REAL HTTP request!"""
    try:
        headers = {"User-Agent": "ISOM260-ResearchAgent/1.0"}
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for element in soup(['script', 'style', 'nav', 'footer', 'header']):
            element.decompose()
        text = soup.get_text(separator='\n', strip=True)
        if len(text) > 3000:
            text = text[:3000] + "\n...[truncated]"
        title = soup.title.string if soup.title else "No title"
        return json.dumps({"title": title, "url": url, "content": text}, indent=2)
    except requests.exceptions.Timeout:
        return json.dumps({"error": f"Request to {url} timed out."})
    except requests.exceptions.HTTPError as e:
        return json.dumps({"error": f"HTTP error {e.response.status_code} for {url}"})
    except Exception as e:
        return json.dumps({"error": f"Failed to fetch {url}: {str(e)}"})


# --- Tool 3: Calculator (REAL) ---
calculator_tool = {
    "name": "calculator",
    "description": (
        "Perform mathematical calculations. Accepts a math expression as a string "
        "and returns the result. Supports +, -, *, /, **, and parentheses."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The math expression to evaluate, e.g. '(42 * 1.5) + 100'"
            }
        },
        "required": ["expression"]
    }
}

def calculator(expression: str) -> str:
    """Evaluate a math expression safely."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": "Invalid characters in expression"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": f"Calculation failed: {str(e)}"})


# Register tools for easy reference
research_tools = [wikipedia_tool, fetch_tool, calculator_tool]
research_functions = {
    "wikipedia_search": wikipedia_search,
    "fetch_web_page": fetch_web_page,
    "calculator": calculator
}

print("✅ 3 Session 5 tools loaded!")
print("   📚 wikipedia_search — Real Wikipedia API")
print("   🌐 fetch_web_page  — Real HTTP requests")
print("   🔢 calculator       — Math expressions")
print("\n   All three use REAL APIs — live data, not simulated!")

In [ ]:
# Quick test — make sure your real tools still work!
print("🔍 Testing wikipedia_search...")
result = wikipedia_search("Artificial intelligence")
data = json.loads(result)
print(f"   Title: {data.get('title', 'ERROR')}")
print(f"   Summary: {data.get('summary', 'ERROR')[:150]}...")
print(f"   ✅ Wikipedia API works!\n")

print("🌐 Testing fetch_web_page...")
result = fetch_web_page("https://en.wikipedia.org/wiki/Suffolk_University")
data = json.loads(result)
print(f"   Title: {data.get('title', 'ERROR')}")
print(f"   Content length: {len(data.get('content', ''))} chars")
print(f"   ✅ Web fetcher works!\n")

print("🔢 Testing calculator...")
result = calculator("(2024 - 1906)")
data = json.loads(result)
print(f"   2024 - 1906 = {data.get('result', 'ERROR')}")
print(f"   ✅ Calculator works!")

---

## 🤔 Part 2: Three Levels of Agent Intelligence

Let's see why the system prompt matters so much. We'll test the **same question** three ways:

1. **Level 0: No tools** — Just Claude generating text (hallucination risk!)
2. **Level 1: Tools + simple prompt** — Your Session 5 agent (finds data but doesn't think deeply)
3. **Level 2: Tools + ReAct prompt** — Today's upgrade (thinks, evaluates, plans)

Same question. Same tools (for Levels 1 & 2). Watch how different the behavior is.

In [ ]:
# ============================================
# LEVEL 0: No tools — just ask Claude directly
# ============================================
# This is what most people do with ChatGPT — no tools, no verification.

def naive_agent(question: str) -> str:
    """Ask Claude without any tools. Just raw text generation."""
    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=1024,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

print("✅ Naive agent defined (no tools, no reasoning framework)")

In [ ]:
# The question we'll test at ALL THREE levels
RESEARCH_QUESTION = (
    "How is artificial intelligence affecting employment? "
    "I need specific data, credible sources, and a balanced analysis."
)

print("="*60)
print("💤 LEVEL 0: Naive Agent (no tools, no reasoning)")
print("="*60)
naive_result = naive_agent(RESEARCH_QUESTION)
print(naive_result[:800])
if len(naive_result) > 800:
    print("\n...[truncated]")
print("\n⚠️  Looks convincing — but where did those numbers come from?")
print("   Are they real? Are they current? No way to verify.")

In [ ]:
# ============================================
# LEVEL 1: Your Session 5 Agent
# ============================================
# Same run_agent() from Session 5 — real tools, simple system prompt.
# This is EXACTLY the code you wrote last session.

def run_agent(user_message: str, tools: list, tool_functions: dict, verbose=True):
    """Session 5's agent — real tools, simple prompt, no reasoning framework."""
    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"{'='*60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while True:
        step += 1
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1024,
            # ⬇️ Session 5's SIMPLE system prompt
            system="You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning.",
            tools=tools,
            messages=messages
        )

        if verbose:
            print(f"\n🔄 Step {step} | Stop reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "text" and verbose:
                    print(f"   🧠 Thinking: {block.text[:200]}...")
                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    if verbose:
                        print(f"   🛠️  Calling: {tool_name}({json.dumps(tool_input)[:100]})")
                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = f"Error: Unknown tool '{tool_name}'"
                    if verbose:
                        print(f"      Result: {result[:200]}...")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
        else:
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text
            if verbose:
                print(f"\n🤖 Answer:\n{final_answer[:600]}")
                if len(final_answer) > 600:
                    print("\n...[truncated]")
                print(f"\n✅ Done in {step} step(s)")
            return final_answer

        if step > 10:
            return "Error: Max steps exceeded."

print("✅ Session 5 agent defined (tools + simple prompt)")

In [ ]:
# ============================================
# LEVEL 1: Session 5 agent — same question, real tools
# ============================================
print("="*60)
print("🔧 LEVEL 1: Session 5 Agent (real tools + simple prompt)")
print("="*60)

session5_result = run_agent(RESEARCH_QUESTION, research_tools, research_functions)

print("\n" + "❗ "*20)
print("NOTICE: The agent used real tools and found real data.")
print("But did it PLAN its research? Did it evaluate source quality?")
print("Did it search for multiple perspectives? Probably not.")
print("It just grabbed the first thing it found and reported it.")

### 🔍 Spot the Difference

| | Level 0: No Tools | Level 1: Session 5 Agent | Level 2: ReAct Agent (next!) |
|---|---|---|---|
| **Data source** | Hallucinated from training | Real Wikipedia + web pages | Same real data, but EVALUATED |
| **Source checking** | None | None — trusts whatever it finds | Reasons about credibility |
| **Research depth** | One-shot generation | Usually 1-2 tool calls | 3-5 steps with planning |
| **Transparency** | Black box | Shows tool calls | Shows THINKING at every step |
| **The code** | No tools | `run_agent()` + simple prompt | Same `run_agent()` + **ReAct prompt** |

The gap between Level 1 and Level 2 is **not the code — it's the system prompt.** Let's prove it.

---

## 🧠 Part 3: Wire the ReAct Loop

Remember your `run_agent()` from Session 5? It had this core structure:

```python
# Session 5's run_agent() — the loop you already know
while step < max_steps:
    response = client.messages.create(model=..., tools=tools, messages=messages)
    if response.stop_reason == "tool_use":
        # execute tool, append result
    else:
        # return final answer
```

Today's `run_react_agent()` has the **exact same loop**. The only difference is the system prompt:

```python
# Session 6's upgrade — just ONE paragraph changes everything
system_prompt = (
    "You are a research agent that uses the ReAct (Reason + Act) framework. "
    "For every task: 1. THINK → 2. ACT → 3. OBSERVE → 4. REPEAT or CONCLUDE."
)
```

**The loop is identical. The system prompt is the difference.**

This is what NanoClaw was doing when it researched live for you in Session 5. Same concept — instruct the LLM to *reason before acting*, and suddenly the agent plans, evaluates, and self-corrects.

```
┌───────────────────────────────────────┐
│ ReAct Loop                              │
│                                         │
│  1. THOUGHT: "I need to find X..."      │
│     ↓                                    │
│  2. ACTION: wikipedia_search("X")       │
│     ↓                                    │
│  3. OBSERVATION: [results]               │
│     ↓                                    │
│  4. THOUGHT: "Source Y looks credible.   │
│     Let me dig deeper..."                │
│     ↓                                    │
│  5. ACTION: fetch_web_page("Y.com")     │
│     ↓                                    │
│  6. OBSERVATION: [full article text]     │
│     ↓                                    │
│  7. THOUGHT: "I have enough info.        │
│     Let me synthesize."                   │
│     ↓                                    │
│  8. FINAL ANSWER                         │
└───────────────────────────────────────┘
```

In [ ]:
# ============================================
# THE ReAct AGENT LOOP
# ============================================
# This is the same core pattern as your Session 5 run_agent(), but upgraded with:
#   1. A ReAct-style system prompt that forces explicit reasoning
#   2. Verbose step-by-step trace printing
#   3. Formatted Thought / Action / Observation labels

def run_react_agent(user_message: str, tools: list, tool_functions: dict, max_steps: int = 10):
    """
    Run a ReAct research agent.

    ReAct = Reason + Act. The agent thinks before every action,
    observes the result, and reflects before deciding the next step.

    This is the exact pattern used by NanoClaw, Claude Code, Perplexity,
    and every production AI agent.
    """
    # The ReAct system prompt — this is what makes the agent THINK
    system_prompt = (
        "You are a research agent that uses the ReAct (Reason + Act) framework. "
        "For every task, follow this pattern:\n\n"
        "1. THINK: Before each action, explain your reasoning. What do you know? "
        "What do you need to find out? Which tool is best for this?\n"
        "2. ACT: Use a tool to gather information.\n"
        "3. OBSERVE: After receiving results, reflect on what you learned. "
        "Is this source credible? Do you need more information?\n"
        "4. REPEAT or CONCLUDE: Either search for more information or synthesize your findings.\n\n"
        "Always think step by step. Evaluate source credibility before trusting data. "
        "When you have enough high-quality information, provide a clear, structured answer "
        "with specific data points and source attributions.\n\n"
        "Be thorough but efficient. Aim for 3-5 research steps before synthesizing."
    )

    print(f"\n{'='*60}")
    print(f"💬 User: {user_message}")
    print(f"{'='*60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1

        # Send message to Claude with tools and ReAct system prompt
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        print(f"\n{'- '*30}")
        print(f"🔄 Step {step} | Stop reason: {response.stop_reason}")

        # Check if Claude wants to use a tool
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text" and block.text.strip():
                    # This is the THOUGHT step — Claude's reasoning
                    print(f"\n   💭 THOUGHT:")
                    for line in block.text.strip().split('\n'):
                        print(f"      {line}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    # This is the ACTION step
                    print(f"\n   🎯 ACTION: {tool_name}")
                    print(f"      Input: {json.dumps(tool_input, indent=2)[:200]}")

                    # Execute the tool
                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                    # This is the OBSERVATION step
                    print(f"\n   👁️  OBSERVATION:")
                    result_preview = result[:300] + "..." if len(result) > 300 else result
                    for line in result_preview.split('\n')[:8]:
                        print(f"      {line}")
                    if len(result) > 300:
                        print(f"      [...truncated for display, full data sent to agent]")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Add to conversation history
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            # Claude is done — this is the FINAL ANSWER
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            print(f"\n{'='*60}")
            print(f"🤖 FINAL ANSWER:")
            print(f"{'='*60}")
            print(final_answer)
            print(f"\n✅ Research complete in {step} step(s)")

            return {"answer": final_answer, "messages": messages, "steps": step}

    # Safety: max steps reached
    print(f"\n⚠️ Max steps ({max_steps}) reached. Returning partial results.")
    return {"answer": "Max steps reached.", "messages": messages, "steps": step}

print("✅ ReAct agent loop defined!")
print("   Same loop as Session 5. Different system prompt. That's it.")

---

## 🔬 Part 4: The Moment of Truth

Time to run the **same question** through the ReAct agent with your **same Session 5 tools**.

Watch the reasoning trace carefully. Compare it to what the Session 5 agent did above. Same tools. Same question. Different behavior.

In [ ]:
# ============================================
# LEVEL 2: ReAct Agent — SAME question, SAME tools
# ============================================
# The ONLY difference from the Session 5 agent above:
# the system prompt tells the agent to THINK before ACTING.

print("="*60)
print("🧠 LEVEL 2: ReAct Agent (same tools + ReAct prompt)")
print("="*60)

react_result_1 = run_react_agent(
    RESEARCH_QUESTION,
    research_tools,
    research_functions
)

print("\n" + "🌟 "*20)
print(f"Session 5 agent: probably 1-2 steps, grabbed first result.")
print(f"ReAct agent: {react_result_1['steps']} steps with visible reasoning.")
print("Same tools. Same question. One paragraph changed everything.")

In [ ]:
# ============================================
# TEST 2: Try a different research question
# ============================================
# Watch the ReAct agent plan a completely different research strategy.
# It adapts its approach based on the question!

react_result_2 = run_react_agent(
    "Compare the electric vehicle markets in the US and China. "
    "Which country is ahead and why? Include specific market data.",
    research_tools,
    research_functions
)

print("\n" + "🎯 "*20)
print("Notice how the agent adapted its research strategy.")
print("It searched for BOTH markets and compared them — a Session 5 agent wouldn't.")

### 🌟 What Just Happened

Compare the three levels. The ReAct agent:

1. **Planned** its research strategy before searching (not just grabbing the first thing)
2. **Searched** for real data from Wikipedia and web pages (not hallucinating)
3. **Evaluated** what it found before trusting it (is this source credible? is this enough?)
4. **Followed up** with additional searches when the first results weren't sufficient
5. **Synthesized** a structured answer with source attribution
6. **Showed its reasoning** at every step (you can audit the entire process)

This is exactly what makes tools like **ChatGPT Deep Research** and **Perplexity** so much better than basic chatbots — they don't just generate text, they **research then generate**.

**This is the exact same pattern NanoClaw used when it researched for you live in Session 5.** The only difference? NanoClaw connects to even more tools via MCP.

And remember: the code change from your Session 5 agent was **one paragraph** — the system prompt.

---

## 🔥 Push the Limits: When Does ReAct Fail?

A research agent that only works on easy questions isn't useful. Let's find the **failure modes** with real data. Try each of these and **watch what happens in the reasoning trace:**

### Challenges:
1. 🏭 **No data exists**: Ask about something Wikipedia doesn't have
2. 🤔 **Ambiguous question**: Ask something too vague for research
3. 🔄 **Recovery**: When a search fails, does the agent try a different approach?
4. ✅ **Knowing when to stop**: Does the agent keep searching forever or synthesize?

**What to watch for:**
- Does the agent handle Wikipedia 404s gracefully? (These are REAL 404s!)
- Does it try alternative search terms when the first one fails?
- Does it admit when data is weak or missing?
- Does the reasoning trace help you diagnose what went wrong?

In [ ]:
# ============================================
# FAILURE TEST 1: Something Wikipedia doesn't have
# ============================================
# This is a REAL test — the Wikipedia API will return a real 404.
# The agent should recognize there's no data and adapt.
# Bad agents hallucinate. Good agents admit uncertainty.

fail_result_1 = run_react_agent(
    "Research the market performance of QuantumLeaf Technologies Inc. "
    "I need their revenue, growth rate, and competitive position.",
    research_tools,
    research_functions
)

print("\n" + "🔍 "*20)
print("DEBRIEF: Did the agent admit it couldn't find real data?")
print("Or did it hallucinate fake numbers? Check the reasoning trace above.")
print("Did it try alternative search terms after the first one failed?")

In [ ]:
# ============================================
# FAILURE TEST 2: Vague question — can the agent handle it?
# ============================================
fail_result_2 = run_react_agent(
    "Is AI good or bad?",
    research_tools,
    research_functions
)

print("\n" + "🔍 "*20)
print("DEBRIEF: How did the agent handle an opinion question?")
print("Did it try to find 'data' for something that's a judgment call?")
print("Did the reasoning trace show it wrestling with the ambiguity?")

# ============================================
# FAILURE TEST 3: YOUR TURN — break the agent!
# ============================================
# What question would expose a weakness? Try it!
# Ideas:
#   - Ask about a very recent event (Wikipedia might not have it yet)
#   - Ask about something in a non-English language
#   - Ask something that requires comparing 5+ sources
#   - Ask for a specific number that doesn't exist anywhere

# my_breaking_question = "..."
# fail_result_3 = run_react_agent(my_breaking_question, research_tools, research_functions)

In [ ]:
# ============================================
# BONUS: Generate a Structured Research Report
# ============================================
# Takes the agent's full conversation and asks Claude to
# write a professional briefing document.

def generate_report(agent_result: dict, topic: str) -> str:
    """
    Takes the full agent conversation and generates a structured
    research briefing with executive summary, findings, and recommendations.
    """
    conversation_text = ""
    for msg in agent_result["messages"]:
        if msg["role"] == "assistant" and isinstance(msg["content"], list):
            for block in msg["content"]:
                if hasattr(block, "text") and block.text:
                    conversation_text += block.text + "\n"
        elif msg["role"] == "assistant" and isinstance(msg["content"], str):
            conversation_text += msg["content"] + "\n"

    conversation_text += "\nFINAL AGENT RESPONSE:\n" + agent_result["answer"]

    report_prompt = f"""Based on the following research agent conversation about "{topic}",
write a structured research briefing.

AGENT CONVERSATION:
{conversation_text}

Write the briefing with this EXACT structure:

# Research Briefing: [Topic]

## Executive Summary
(2-3 sentences capturing the key finding)

## Key Findings
(Numbered list of 4-6 specific findings with data points)

## Source Quality Assessment
(Brief assessment of the sources used and their reliability)

## Confidence Level
(High/Medium/Low with justification)

## Recommended Next Steps
(3-4 actionable recommendations)

Keep it concise and professional. Every claim should reference a source."""

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2048,
        messages=[{"role": "user", "content": report_prompt}]
    )

    return response.content[0].text

print("✅ Report generator defined!")

In [ ]:
# ============================================
# Generate a report from the AI employment research
# ============================================

print("📝 Generating structured research briefing...")
print("="*60)

report = generate_report(react_result_1, "Impact of AI on Employment")
print(report)

print("\n" + "="*60)
print("🌟 This is what a ReAct research agent produces:")
print("   → Structured, sourced, confidence-rated research.")
print("   → Not a generic essay — an actionable briefing.")
print("   → Built from REAL Wikipedia data, not hallucinated.")

---

## 🎨 Part 5: Build Your Own Tool (Challenge)

Your research agent has 3 tools. But real research agents need more. Your challenge: **add a 4th tool** to make the agent even more powerful.

### Worked Example: Real Weather Tool

Here's a complete 4th tool using a **real API** (free, no key needed) so you can see the full pattern:

1. **Tool definition** — tells Claude what it can do
2. **Python function** — the actual API call
3. **Register** — add to the tools list
4. **Test** — ask a question that triggers it

### Then Build Yours! Starter Ideas:

| Tool Idea | Real API | What It Returns |
|-----------|----------|------------------|
| `get_weather` | Open-Meteo (free) | Temperature, wind, conditions for any city |
| `country_info` | REST Countries (free) | Population, GDP, languages, region |
| `random_fact` | Numbers API (free) | Fun facts about numbers (for context) |
| Your own idea | Any free API | Whatever makes your agent smarter |

Pick one (or invent your own!) and follow the same pattern below:

In [ ]:
# ============================================
# WORKED EXAMPLE: Real Weather Tool (free, no API key!)
# ============================================
# Uses Open-Meteo API: geocodes a city name, then gets current weather.
# Run this cell to see it work, then build your own below.

weather_tool = {
    "name": "get_weather",
    "description": (
        "Get the current weather for any city in the world. Returns temperature, "
        "wind speed, and conditions. Use this when weather context is relevant "
        "to the research (e.g., agriculture, travel, energy, outdoor events)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. 'Boston', 'Tokyo', 'London'"
            }
        },
        "required": ["city"]
    }
}

def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo API (free, no key needed)."""
    try:
        # Step 1: Convert city name to coordinates
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_resp = requests.get(geo_url, timeout=10)
        geo_data = geo_resp.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return json.dumps({"error": f"City '{city}' not found"})

        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        city_name = geo_data["results"][0]["name"]
        country = geo_data["results"][0].get("country", "Unknown")

        # Step 2: Get current weather
        weather_url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}&current_weather=true"
        )
        weather_resp = requests.get(weather_url, timeout=10)
        weather_data = weather_resp.json()["current_weather"]

        return json.dumps({
            "city": city_name,
            "country": country,
            "temperature_celsius": weather_data["temperature"],
            "windspeed_kmh": weather_data["windspeed"],
            "weather_code": weather_data.get("weathercode", 0),
        }, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Weather lookup failed: {str(e)}"})


# Register ALL 4 tools (3 from Session 5 + your new one)
all_tools = [wikipedia_tool, fetch_tool, calculator_tool, weather_tool]
all_functions = {
    "wikipedia_search": wikipedia_search,
    "fetch_web_page": fetch_web_page,
    "calculator": calculator,
    "get_weather": get_weather
}

# Test it! This query should trigger the new weather tool
weather_result = run_react_agent(
    "I'm planning a business trip to Tokyo next week. "
    "What's the weather like there? Also look up Tokyo on Wikipedia "
    "and tell me about the business district.",
    all_tools,
    all_functions
)

print("\n" + "🎯 "*20)
print("Did the agent use BOTH Wikipedia AND your new weather tool?")
print("That's the power of adding tools — the agent figures out when to use each one.")

In [ ]:
# ============================================
# YOUR TURN: Build your own 4th tool!
# ============================================
# Follow the same pattern as get_weather above.
# Use a REAL API if you can! Ideas:
#   - REST Countries: https://restcountries.com/v3.1/name/{name}
#   - Numbers API: http://numbersapi.com/{number}
#   - Open-Meteo: https://open-meteo.com (more weather options)
#   - Or any other free, no-key API you find!

# Step 1: Tool definition (tells Claude what this tool can do)
my_tool = {
    "name": "your_tool_name",             # <-- Change this
    "description": (
        "Describe what your tool does. Be detailed! "  # <-- Change this
        "Tell Claude when to use it and what it returns."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "param1": {                      # <-- Change parameter name
                "type": "string",
                "description": "What this parameter means"  # <-- Change this
            }
        },
        "required": ["param1"]
    }
}

# Step 2: Python function (use requests.get for real APIs!)
def your_tool_name(param1: str) -> str:    # <-- Match the tool name!
    """Your tool logic here. Use requests.get() for a real API!"""
    try:
        # url = f"https://your-api-here.com/{param1}"
        # response = requests.get(url, timeout=10)
        # data = response.json()
        # return json.dumps(data, indent=2)
        return json.dumps({"error": "Not implemented yet — replace this!"})
    except Exception as e:
        return json.dumps({"error": str(e)})


# Step 3: Register all tools (including your new one)
my_tools = [wikipedia_tool, fetch_tool, calculator_tool, my_tool]
my_functions = {
    "wikipedia_search": wikipedia_search,
    "fetch_web_page": fetch_web_page,
    "calculator": calculator,
    "your_tool_name": your_tool_name          # <-- Match the tool name!
}

# Step 4: Test with 3 questions that should trigger your tool
# run_react_agent("Your test question 1", my_tools, my_functions)
# run_react_agent("Your test question 2", my_tools, my_functions)
# run_react_agent("Your test question 3", my_tools, my_functions)

# 🤝 Compare results with a neighbor — whose 4th tool is more useful?

---

## 💭 Part 6: Reflection — The Bigger Picture

### What You Built vs. Production Systems

| What You Built Today | Production Systems (Perplexity, Claude Code, NanoClaw) |
|---|---|
| 3 real tools (Wikipedia, web fetch, calculator) | Hundreds of tools via MCP |
| Single-agent ReAct loop | Multi-agent pipelines: Researcher → Writer → Editor |
| Reasoning printed to console | Reasoning stored, auditable, used for improvement |
| Real Wikipedia + web data | Real search engines + databases + proprietary data |
| ~80 lines of agent code | Thousands of lines + infrastructure |
| Runs in Colab for class | Runs 24/7 on servers, handles millions of queries |

### But the Core Pattern Is Identical

```
1. Receive a research question
2. THINK about what information is needed
3. ACT by calling the right tool
4. OBSERVE the results and evaluate quality
5. REPEAT until you have enough high-quality data
6. SYNTHESIZE into a structured answer
```

You built the same architecture. Production just adds:
- **More tools** connected via MCP (the "USB-C of AI")
- **Multiple agents** collaborating (researcher + writer + editor)
- **Persistent memory** across sessions
- **Error handling** and retries for reliability
- **Security** and sandboxing for safety

### 💡 The Business Insight

The **most valuable skill** isn't coding the ReAct loop (that's ~80 lines). It's:

1. **Choosing what to research** — Which business questions benefit from agent-powered research?
2. **Designing the right tools** — What data sources matter? What's the evaluation criteria?
3. **Evaluating the output** — Is the agent's research actually *better* than a human analyst?
4. **Knowing when to stop** — When does more automation hurt rather than help?

These are business decisions, not engineering decisions. That's why **you** — as business students — are perfectly positioned to design the next generation of AI agents.

### Real-World Examples Using This Exact Pattern

- **NanoClaw** (built for this class) uses ReAct with MCP tools — the same loop you just ran
- **Claude Code** reasons through multi-step coding tasks with think-act-observe
- **Perplexity** searches, evaluates, and synthesizes — exactly what your agent does
- **Devin** (AI software engineer) uses ReAct to plan, code, test, and debug

You now understand the architecture behind all of them.

---

## 🚀 What's Next?

- **Homework**: Add a 4th tool (real API!) to your agent. Test it with 3 research questions. Document the reasoning traces.
- **Session 7**: Knowledge Agents — RAG. Your agent is smart, but it doesn't know YOUR business. Next session: ground it in company documents.

---

**ISOM 260: AI for Business** | Suffolk University | Session 6

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)